In [16]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path(r'C:\Users\gordi\Desktop\Transmilenio\Proyecto_Investigacion_Teorica_2026_2\datos')

# Un archivo de entrada y salida por año
MUESTRA = {
    2019: ('ENERO19.xlsx',    'ABRIL19S.xlsx'),
    2021: ('ENERO21.xlsx',    'ENERO21S.xlsx'),
    2022: ('FEBRERO22.xlsx',  'FEBRERO22S.xlsx'),
    2023: ('MARZO23.xlsx',    'MARZO23S.xlsx'),
    2024: ('ABRIL24.xlsx',    'ABRIL24S.xlsx'),
    2025: ('MAYO25.xlsx',     'MAYO25S.xlsx'),
}

def inspeccionar(fp: Path, tipo: str):
    if not fp.exists():
        print(f"  NO EXISTE: {fp.name}")
        return

    raw = pd.read_excel(fp, header=None, nrows=15, dtype=str)
    header_row = 6
    for i, row in raw.iterrows():
        if any(str(v).strip() in ('Estación','ESTACION','Estacion') for v in row if pd.notna(v)):
            header_row = i
            break

    df = pd.read_excel(fp, skiprows=header_row, header=0, nrows=4, dtype=str)
    df = df.loc[:, ~df.columns.astype(str).str.contains("Total", case=False)]

    cols = list(df.columns)
    print(f"  Header en fila {header_row} | {len(cols)} columnas")
    print(f"  Primeras 4 cols: {[repr(c) for c in cols[:4]]}")
    print(f"  Últimas 3 cols:  {[repr(c) for c in cols[-3:]]}")
    print(f"  Fila muestra:")
    print(df.iloc[2:3].to_string(index=False))

for año, (e, s) in MUESTRA.items():
    print(f"\n{'─'*55}")
    print(f"  {año} — ENTRADAS: {e}")
    inspeccionar(DATA_DIR / e, 'entradas')
    print(f"  {año} — SALIDAS:  {s}")
    inspeccionar(DATA_DIR / s, 'salidas')


───────────────────────────────────────────────────────
  2019 — ENTRADAS: ENERO19.xlsx
  Header en fila 6 | 37 columnas
  Primeras 4 cols: ["'Unnamed: 0'", "'Fase'", "'Línea'", "'Estación'"]
  Últimas 3 cols:  ['datetime.datetime(2019, 1, 29, 0, 0)', 'datetime.datetime(2019, 1, 30, 0, 0)', 'datetime.datetime(2019, 1, 31, 0, 0)']
  Fila muestra:
Unnamed: 0   Fase                Línea                Estación                                                                                    Acceso de Estación Intervalo 2019-01-01 00:00:00 2019-01-02 00:00:00 2019-01-03 00:00:00 2019-01-04 00:00:00 2019-01-05 00:00:00 2019-01-06 00:00:00 2019-01-07 00:00:00 2019-01-08 00:00:00 2019-01-09 00:00:00 2019-01-10 00:00:00 2019-01-11 00:00:00 2019-01-12 00:00:00 2019-01-13 00:00:00 2019-01-14 00:00:00 2019-01-15 00:00:00 2019-01-16 00:00:00 2019-01-17 00:00:00 2019-01-18 00:00:00 2019-01-19 00:00:00 2019-01-20 00:00:00 2019-01-21 00:00:00 2019-01-22 00:00:00 2019-01-23 00:00:00 2019-01-24 00:00

In [17]:
import pandas as pd
import random
from pathlib import Path

PARQUET_DIR = Path(r'C:\Users\gordi\Desktop\Transmilenio\Proyecto_Investigacion_Teorica_2026_2\outputs\parquet')

años = [2019, 2021, 2022, 2023, 2024, 2025]

for año in años:
    print(f"\n{'─'*60}")
    print(f"  {año}")
    print(f"{'─'*60}")

    for tipo in ['entradas', 'salidas']:
        archivos = sorted(PARQUET_DIR.glob(f"{año}-*-{tipo}.parquet"))
        if len(archivos) < 2:
            print(f"  [WARN] {tipo}: menos de 2 archivos disponibles")
            continue

        muestra = random.sample(archivos, 2)
        for fp in muestra:
            df = pd.read_parquet(fp)
            df_op = df[df['datetime'].dt.hour.between(6, 22)]
            fila  = df_op.sample(1).iloc[0]
            print(f"  {fp.name}")
            print(f"    estacion={fila['station_id']}  linea={fila['linea_id']}"
                  f"  datetime={fila['datetime']}  {tipo}={fila[tipo]}")


────────────────────────────────────────────────────────────
  2019
────────────────────────────────────────────────────────────
  2019-05-entradas.parquet
    estacion=07104  linea=00038  datetime=2019-05-27 22:45:00  entradas=17
  2019-06-entradas.parquet
    estacion=09105  linea=00034  datetime=2019-06-20 21:00:00  entradas=48
  2019-09-salidas.parquet
    estacion=03013  linea=00032  datetime=2019-09-01 17:30:00  salidas=18
  2019-05-salidas.parquet
    estacion=06000  linea=00011  datetime=2019-05-01 13:45:00  salidas=436

────────────────────────────────────────────────────────────
  2021
────────────────────────────────────────────────────────────
  2021-05-entradas.parquet
    estacion=07101  linea=00038  datetime=2021-05-03 12:00:00  entradas=20
  2021-11-entradas.parquet
    estacion=02104  linea=00033  datetime=2021-11-08 13:00:00  entradas=65
  2021-05-salidas.parquet
    estacion=40004  linea=00040  datetime=2021-05-10 08:15:00  salidas=0
  2021-08-salidas.parquet
    es